# CEVAE実験 v5：真のZ=2次元でのリベンジ + 崩壊ガード付き選択

**背景**：1次元Zの実験(v4)で、CEVAEのrunは「成功(√PEHE≈1.3-1.7)」と「崩壊(√PEHE≈6-8)」の二山に割れ、主因は初期値運（posterior collapse。zが交絡でなく処置Tを符号化してしまう）と分かった。val_ELBOだけでは二山を区別できない（崩壊runのELBOが最良になることがある＝非識別性）。一方で `mse_ratio = mse_p/mse_q`（zを通すとyの予測誤差が何倍悪化するか）は、真のτ・Zを使わずに二山を分離できた。

**このノートブックでやること**：真のZを2次元に戻し（設計書§2準拠、v3のDGPを流用）、
1. z_dimもランダムサーチで探す（真の次元=2はモデルに教えない）
2. val_ELBOでの選択に「mse_ratio崩壊ガード」を追加する（外れ値のtrialを弾いてからELBO最良を選ぶ）
3. 採用設定で複数data_seed×複数model_seed(再挑戦)を回し、崩壊ガードで濾過した後にX-learnerと比較する

真のτ・真のZは「採点」（√PEHE, z_recovery列）にのみ使い、ランダムサーチ・z_dim選択・崩壊ガードには**絶対に使わない**。

既存の v2/v3/v4 は変更しない。このノートブックは単体で上から実行すれば完結する。

**想定実行時間**：ランダムサーチ(N_ITER=40)で約10分、マルチシード実験(10 data_seed×5 model_seed=50 run)で約12分、X-learner探索で数分。合計で**30〜40分程度**を見込む（重い処理なので、Kernelを繋いだまま最後まで実行し切ることを推奨）。

In [1]:
import warnings
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.stats import uniform, randint
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", message=".*encountered in matmul.*", category=RuntimeWarning)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def generate_toy_v5(N=20000, d=8, sigma_x=3.5, sigma_y=1.0, seed=0):
    """v5 DGP: 真のZは2次元。v3の generate_toy_v3 をそのまま流用（設計書§2準拠＋非線形項）。
    Z = [Z1, Z2] ~ N(0, I2)
    X = tanh(Z @ A) + noise,  A ∈ R^{2×d} ~ N(1,1)
    T | Z ~ Bernoulli(σ(1.5*Z1 + 0.5*Z2))
    τ(Z) = 1.0 + Z1 + 0.5*Z2 + 0.8*(Z1²+Z2²-2)
    Y0 = 2.0*Z1 + 1.0*Z2 + noise,  Y1 = Y0 + τ(Z)
    """
    rng = np.random.default_rng(seed)
    Z = rng.normal(0, 1, size=(N, 2))
    A = rng.normal(1.0, 1.0, size=(2, d))
    X = np.tanh(Z @ A) + rng.normal(0, sigma_x, size=(N, d))
    w_T = np.array([1.5, 0.5])
    T = rng.binomial(1, sigmoid(Z @ w_T))
    w_tau = np.array([1.0, 0.5])
    tau = 1.0 + Z @ w_tau + 0.8 * (np.sum(Z ** 2, axis=1) - 2)
    w_Y = np.array([2.0, 1.0])
    base = Z @ w_Y
    Y0 = base + rng.normal(0, sigma_y, size=N)
    Y1 = base + tau + rng.normal(0, sigma_y, size=N)
    Y = np.where(T == 1, Y1, Y0)
    return X, T, Y, Y0, Y1, tau, Z

In [2]:
DATA_SEED = 0
X, T, Y, Y0, Y1, tau, Z = generate_toy_v5(seed=DATA_SEED)
d = X.shape[1]
N = len(Y)

proxy_cols = [f"X{j+1}" for j in range(d)]
df_full = pd.DataFrame(X, columns=proxy_cols)
df_full["T"] = T; df_full["Y"] = Y; df_full["tau"] = tau

df_train, df_test = train_test_split(df_full, test_size=0.3,
                                     random_state=DATA_SEED, stratify=df_full["T"])

print(f"N={N}, d={d}, train={len(df_train)}, test={len(df_test)}")
print(f"真のATE       : {tau.mean():.3f}")
print(f"ナイーブATE   : {Y[T==1].mean()-Y[T==0].mean():.3f}")
print(f"corr(T, Z1)   : {np.corrcoef(T, Z[:,0])[0,1]:.3f}")
print(f"corr(T, Z2)   : {np.corrcoef(T, Z[:,1])[0,1]:.3f}")
Xd = np.column_stack([np.ones(N), X])
for j in range(2):
    coef, _, _, _ = np.linalg.lstsq(Xd, Z[:, j], rcond=None)
    r2 = 1 - np.sum((Z[:, j] - Xd @ coef) ** 2) / np.sum((Z[:, j] - Z[:, j].mean()) ** 2)
    print(f"R^2(Z{j+1}|X) 線形回帰 : {r2:.3f}")

N=20000, d=8, train=14000, test=6000
真のATE       : 1.011
ナイーブATE   : 4.012
corr(T, Z1)   : 0.520
corr(T, Z2)   : 0.169
R^2(Z1|X) 線形回帰 : 0.162
R^2(Z2|X) 線形回帰 : 0.091


## X-learner ハイパラ探索（data_seed=0）

In [3]:
XGB_PARAM_DIST = {
    "n_estimators": randint(300, 2000),
    "learning_rate": uniform(0.01, 0.19),
    "max_depth": randint(2, 8),
    "min_child_weight": randint(1, 10),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
    "reg_lambda": uniform(0.0, 10.0),
}

def search_xgb(X_in, y_in, n_iter=30, cv=3, seed=0):
    s = RandomizedSearchCV(
        XGBRegressor(objective="reg:squarederror", random_state=seed, verbosity=0),
        param_distributions=XGB_PARAM_DIST,
        n_iter=n_iter, cv=cv, scoring="neg_mean_squared_error",
        n_jobs=-1, random_state=seed,
    )
    s.fit(X_in, y_in)
    return s.best_estimator_

def run_xlearner(df_tr, df_te, proxy_cols, seed=0):
    df0 = df_tr[df_tr["T"] == 0]
    df1 = df_tr[df_tr["T"] == 1]
    m0 = search_xgb(df0[proxy_cols], df0["Y"], seed=seed)
    m1 = search_xgb(df1[proxy_cols], df1["Y"], seed=seed)
    D0 = m1.predict(df0[proxy_cols]) - df0["Y"].values
    D1 = df1["Y"].values - m0.predict(df1[proxy_cols])
    x0 = search_xgb(df0[proxy_cols], D0, seed=seed)
    x1 = search_xgb(df1[proxy_cols], D1, seed=seed)
    prop = LogisticRegression(max_iter=1000).fit(df_tr[proxy_cols], df_tr["T"])
    g0 = prop.predict_proba(df_te[proxy_cols])[:, 0]
    tau_pred = g0 * x0.predict(df_te[proxy_cols]) + (1 - g0) * x1.predict(df_te[proxy_cols])
    return tau_pred, (m0, m1, x0, x1, prop)

print("X-learner ハイパラ探索中（data_seed=0）...")
xl_tau_pred, xl_models_fixed = run_xlearner(df_train, df_test, proxy_cols, seed=0)
tau_true_test_s0 = df_test["tau"].values
xl_pehe_s0 = np.sqrt(np.mean((tau_true_test_s0 - xl_tau_pred) ** 2))
print(f"X-learner √PEHE (data_seed=0): {xl_pehe_s0:.3f}")

X-learner ハイパラ探索中（data_seed=0）...
X-learner √PEHE (data_seed=0): 3.161


## ネットワーク・関数定義（CEVAE本体 + 崩壊ガード用の診断関数）

In [4]:
class DecoderX(nn.Module):
    def __init__(self, z_dim, d, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, d))
    def forward(self, z): return self.net(z)

class DecoderT(nn.Module):
    def __init__(self, z_dim, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, z): return self.net(z).squeeze(-1)

class DecoderY(nn.Module):
    def __init__(self, z_dim, hidden):
        super().__init__()
        self.head0 = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(z_dim, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, z): return self.head0(z).squeeze(-1), self.head1(z).squeeze(-1)

class EncoderT(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x): return self.net(x).squeeze(-1)

class EncoderY(nn.Module):
    def __init__(self, d, hidden):
        super().__init__()
        self.head0 = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
        self.head1 = nn.Sequential(nn.Linear(d, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, x): return self.head0(x).squeeze(-1), self.head1(x).squeeze(-1)

class EncoderZ(nn.Module):
    def __init__(self, d, z_dim, hidden):
        super().__init__()
        self.trunk   = nn.Sequential(nn.Linear(d, hidden), nn.ReLU())
        self.head0   = nn.Sequential(nn.Linear(hidden + 1, hidden), nn.ReLU())
        self.head1   = nn.Sequential(nn.Linear(hidden + 1, hidden), nn.ReLU())
        self.mu0     = nn.Linear(hidden, z_dim)
        self.logvar0 = nn.Linear(hidden, z_dim)
        self.mu1     = nn.Linear(hidden, z_dim)
        self.logvar1 = nn.Linear(hidden, z_dim)
    def forward(self, x, t, y):
        h  = self.trunk(x)
        hy = torch.cat([h, y.unsqueeze(-1)], dim=-1)
        h0, h1 = self.head0(hy), self.head1(hy)
        mu0, lv0 = self.mu0(h0), self.logvar0(h0)
        mu1, lv1 = self.mu1(h1), self.logvar1(h1)
        tc = t.unsqueeze(-1)
        return torch.where(tc==1, mu1, mu0), torch.where(tc==1, lv1, lv0)

def reparameterize(mu, logvar):
    return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

def kl_to_standard_normal(mu, logvar):
    return (-0.5 * (1 + logvar - mu.pow(2) - logvar.exp())).sum(dim=-1).mean()

def linear_kl_anneal(step, total_steps):
    return 1.0 if total_steps <= 0 else min(1.0, step / total_steps)

@dataclass
class CEVAEConfig:
    z_dim: int = 2
    hidden: int = 64
    beta_kl_max: float = 0.3
    kl_anneal_epochs: int = 100
    w_recon_y: float = 1.0
    w_aux_y: float = 1.0
    epochs: int = 200
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 1e-4
    seed: int = 0

def build_cevae(cfg, d):
    torch.manual_seed(cfg.seed)
    return {
        "decoder_x": DecoderX(cfg.z_dim, d, cfg.hidden),
        "decoder_t": DecoderT(cfg.z_dim, cfg.hidden),
        "decoder_y": DecoderY(cfg.z_dim, cfg.hidden),
        "encoder_t": EncoderT(d, cfg.hidden),
        "encoder_y": EncoderY(d, cfg.hidden),
        "encoder_z": EncoderZ(d, cfg.z_dim, cfg.hidden),
    }

def cevae_loss_fn(nets, x, t, y, beta_kl, w_recon_y=1.0, w_aux_y=1.0):
    mu_z, logvar_z = nets["encoder_z"](x, t, y)
    z = reparameterize(mu_z, logvar_z)
    x_hat = nets["decoder_x"](z)
    t_logit_p = nets["decoder_t"](z)
    y0p, y1p = nets["decoder_y"](z)
    y_hat_p = torch.where(t == 1, y1p, y0p)
    recon_x = nn.functional.mse_loss(x_hat, x)
    recon_t = nn.functional.binary_cross_entropy_with_logits(t_logit_p, t)
    recon_y = nn.functional.mse_loss(y_hat_p, y)
    kl = kl_to_standard_normal(mu_z, logvar_z)
    t_logit_q = nets["encoder_t"](x)
    y0q, y1q = nets["encoder_y"](x)
    y_hat_q = torch.where(t == 1, y1q, y0q)
    aux_t = nn.functional.binary_cross_entropy_with_logits(t_logit_q, t)
    aux_y = nn.functional.mse_loss(y_hat_q, y)
    return recon_x + recon_t + w_recon_y*recon_y + beta_kl*kl + aux_t + w_aux_y*aux_y

def train_cevae(nets, cfg, x_tr, t_tr, y_tr):
    torch.manual_seed(cfg.seed)
    all_params = [p for net in nets.values() for p in net.parameters()]
    opt = torch.optim.Adam(all_params, lr=cfg.lr, weight_decay=cfg.weight_decay)
    n = x_tr.shape[0]
    n_batches = max(1, n // cfg.batch_size)
    for epoch in range(cfg.epochs):
        beta_kl = cfg.beta_kl_max * linear_kl_anneal(epoch, cfg.kl_anneal_epochs)
        perm = torch.randperm(n)
        for b in range(n_batches):
            idx = perm[b*cfg.batch_size:(b+1)*cfg.batch_size]
            opt.zero_grad()
            cevae_loss_fn(nets, x_tr[idx], t_tr[idx], y_tr[idx],
                          beta_kl, cfg.w_recon_y, cfg.w_aux_y).backward()
            opt.step()

def compute_val_elbo(nets, x_val, t_val, y_val):
    with torch.no_grad():
        return cevae_loss_fn(nets, x_val, t_val, y_val, beta_kl=1.0).item()

def estimate_ite(nets, x):
    with torch.no_grad():
        t_hat = (torch.sigmoid(nets["encoder_t"](x)) > 0.5).float()
        y0q, y1q = nets["encoder_y"](x)
        y_hat = torch.where(t_hat == 1, y1q, y0q)
        mu_z, _ = nets["encoder_z"](x, t_hat, y_hat)
        y0p, y1p = nets["decoder_y"](mu_z)
        return (y1p - y0p).numpy()


def compute_y_mse(nets, x_obs, t_obs, y_obs_np):
    """真のτ・真のZを使わない、held-outデータだけで計算できる診断指標。
    mse_q: q(y|x,t)（zを介さない、既存の補助NW encoder_y）の予測誤差。
    mse_p: test-timeフロー（zを推定してからdecoder_yで予測。真のyは使わない）の予測誤差。
    mse_ratio = mse_p / mse_q。zを通すとy予測が何倍悪化するか。1に近いほど健全、大きいほど崩壊疑い。
    """
    with torch.no_grad():
        y0_q, y1_q = nets["encoder_y"](x_obs)
        y_q_hat = torch.where(t_obs == 1, y1_q, y0_q)

        t_hat = (torch.sigmoid(nets["encoder_t"](x_obs)) > 0.5).float()
        y0_for_z, y1_for_z = nets["encoder_y"](x_obs)
        y_hat_for_z = torch.where(t_hat == 1, y1_for_z, y0_for_z)
        z_mu, _ = nets["encoder_z"](x_obs, t_hat, y_hat_for_z)
        y0_p, y1_p = nets["decoder_y"](z_mu)
        y_p_hat = torch.where(t_obs == 1, y1_p, y0_p)

    mse_q = float(np.mean((y_obs_np - y_q_hat.numpy()) ** 2))
    mse_p = float(np.mean((y_obs_np - y_p_hat.numpy()) ** 2))
    mse_ratio = mse_p / mse_q if mse_q > 0 else float("inf")
    return mse_q, mse_p, mse_ratio


def z_recovery_score(nets, x, t, y, Z_true):
    """真のZが何次元でも使える、回転不変なz復元度（採点専用。選択には絶対に使わない）。
    真のZの各列を mu_z（全次元）から線形回帰し、決定係数R²の平均を返す（成分ごとcorrは使わない）。
    """
    with torch.no_grad():
        mu_z, _ = nets["encoder_z"](x, t, y)
    mz = mu_z.numpy()
    Zt = Z_true if Z_true.ndim > 1 else Z_true[:, None]
    scores = []
    for j in range(Zt.shape[1]):
        reg = LinearRegression().fit(mz, Zt[:, j])
        scores.append(max(reg.score(mz, Zt[:, j]), 0.0))
    return float(np.mean(scores))


def guard_threshold(mse_ratio_series, k=3.0, min_abs=1.5):
    """mse_ratioの分布から「崩壊」を弾く閾値を決める。
    相対ルール（中央値からの頑健な外れ値検出、MAD基準）を主に、目安として絶対1.5を下限に使う。
    """
    med = mse_ratio_series.median()
    mad = (mse_ratio_series - med).abs().median()
    robust = med + k * 1.4826 * mad
    return max(robust, min_abs)

## CEVAE ランダムサーチ（data_seed=0, val_ELBO + mse_ratio崩壊ガード）

z_dim も探索対象に含める（真の次元=2はモデルに教えない）。各trialで val_ELBO と mse_ratio を記録し、
mse_ratio が明らかな外れ値のtrialを弾いてから val_ELBO 最良を採用する（崩壊ガード）。
ガード無し（val_ELBO単体で最良を選ぶ場合）との違いも比較する。

In [ ]:
df_inner, df_val = train_test_split(df_train, test_size=0.2,
                                    random_state=0, stratify=df_train["T"])

def df_to_tensors(df):
    x = torch.tensor(df[proxy_cols].values, dtype=torch.float32)
    t = torch.tensor(df["T"].values, dtype=torch.float32)
    y = torch.tensor(df["Y"].values, dtype=torch.float32)
    return x, t, y

x_inner, t_inner, y_inner = df_to_tensors(df_inner)
x_val,   t_val,   y_val   = df_to_tensors(df_val)
y_val_np = df_val["Y"].values

SEARCH_SPACE = {
    "z_dim":            [1, 2, 3],
    "hidden":           [32, 64, 128],
    "beta_kl_max":      [0.1, 0.3, 0.5, 1.0],
    "kl_anneal_epochs": [50, 100, 150],
    "epochs":           [150, 300],
    "weight_decay":     [0.0, 1e-4, 1e-3],
    "lr":               [1e-3, 5e-4, 3e-4],
}
N_ITER = 60  # v3(20)より増やす。z_dimを含む探索＋崩壊ガード後に十分な候補を残すため

rng_search = random.Random(42)
search_records = []

print(f"CEVAE ランダムサーチ（n_iter={N_ITER}, z_dimも含めて探索）...")
for i in range(N_ITER):
    cfg = CEVAEConfig(
        z_dim=rng_search.choice(SEARCH_SPACE["z_dim"]),
        hidden=rng_search.choice(SEARCH_SPACE["hidden"]),
        beta_kl_max=rng_search.choice(SEARCH_SPACE["beta_kl_max"]),
        kl_anneal_epochs=rng_search.choice(SEARCH_SPACE["kl_anneal_epochs"]),
        epochs=rng_search.choice(SEARCH_SPACE["epochs"]),
        weight_decay=rng_search.choice(SEARCH_SPACE["weight_decay"]),
        lr=rng_search.choice(SEARCH_SPACE["lr"]),
        seed=0,
    )
    nets = build_cevae(cfg, d)
    train_cevae(nets, cfg, x_inner, t_inner, y_inner)
    val_elbo = compute_val_elbo(nets, x_val, t_val, y_val)
    mse_q, mse_p, mse_ratio = compute_y_mse(nets, x_val, t_val, y_val_np)
    search_records.append({"iter": i, "val_elbo": val_elbo, "mse_ratio": mse_ratio,
                           "mse_q": mse_q, "mse_p": mse_p,
                           "z_dim": cfg.z_dim, "hidden": cfg.hidden,
                           "beta_kl_max": cfg.beta_kl_max,
                           "kl_anneal_epochs": cfg.kl_anneal_epochs,
                           "epochs": cfg.epochs, "weight_decay": cfg.weight_decay,
                           "lr": cfg.lr})
    print(f"  [{i+1:3d}/{N_ITER}] val_elbo={val_elbo:.3f}  mse_ratio={mse_ratio:.3f}  "
          f"z={cfg.z_dim} h={cfg.hidden} β={cfg.beta_kl_max} "
          f"anneal={cfg.kl_anneal_epochs} ep={cfg.epochs} wd={cfg.weight_decay} lr={cfg.lr:.0e}")

df_search = pd.DataFrame(search_records)

# ── ガード無し：val_ELBO最良をそのまま採用 ──
best_row_noguard = df_search.sort_values("val_elbo").iloc[0]

# ── ガード有り：mse_ratioの外れ値を弾いてからval_ELBO最良を採用 ──
guard_th = guard_threshold(df_search["mse_ratio"])
df_search["guard_pass"] = df_search["mse_ratio"] <= guard_th
n_excluded = int((~df_search["guard_pass"]).sum())
df_survive = df_search[df_search["guard_pass"]]
best_row_guard = df_survive.sort_values("val_elbo").iloc[0]

print(f"\n崩壊ガード閾値(mse_ratio) = {guard_th:.3f}  → {n_excluded}/{N_ITER} trial を除外")
print(f"\n[ガード無し] 採用: z_dim={int(best_row_noguard['z_dim'])}  val_elbo={best_row_noguard['val_elbo']:.3f}  "
      f"mse_ratio={best_row_noguard['mse_ratio']:.3f}")
print(f"[ガード有り] 採用: z_dim={int(best_row_guard['z_dim'])}  val_elbo={best_row_guard['val_elbo']:.3f}  "
      f"mse_ratio={best_row_guard['mse_ratio']:.3f}")

BEST_CEVAE_CFG = CEVAEConfig(
    z_dim=int(best_row_guard["z_dim"]),
    hidden=int(best_row_guard["hidden"]),
    beta_kl_max=float(best_row_guard["beta_kl_max"]),
    kl_anneal_epochs=int(best_row_guard["kl_anneal_epochs"]),
    epochs=int(best_row_guard["epochs"]),
    weight_decay=float(best_row_guard["weight_decay"]),
    lr=float(best_row_guard["lr"]),
    seed=0,
)
print(f"\n以降のマルチシード実験で使う採用設定（ガード有り）: {BEST_CEVAE_CFG}")
display(df_search.round(4))

CEVAE ランダムサーチ（n_iter=60, z_dimも含めて探索）...
  [  1/60] val_elbo=26.551  mse_ratio=6.882  z=3 h=32 β=0.1 anneal=150 ep=300 wd=0.0 lr=1e-03
  [  2/60] val_elbo=25.604  mse_ratio=7.554  z=1 h=128 β=0.1 anneal=150 ep=150 wd=0.001 lr=5e-04
  [  3/60] val_elbo=25.680  mse_ratio=4.748  z=1 h=32 β=0.1 anneal=50 ep=150 wd=0.001 lr=3e-04
  [  4/60] val_elbo=25.147  mse_ratio=18.461  z=1 h=128 β=0.3 anneal=150 ep=300 wd=0.0 lr=5e-04
  [  5/60] val_elbo=26.225  mse_ratio=9.589  z=3 h=64 β=0.1 anneal=50 ep=300 wd=0.0001 lr=5e-04
  [  6/60] val_elbo=24.792  mse_ratio=6.663  z=1 h=32 β=0.5 anneal=50 ep=150 wd=0.0001 lr=1e-03
  [  7/60] val_elbo=24.320  mse_ratio=5.221  z=2 h=64 β=0.5 anneal=50 ep=300 wd=0.001 lr=1e-03
  [  8/60] val_elbo=24.188  mse_ratio=5.014  z=2 h=32 β=0.5 anneal=150 ep=300 wd=0.001 lr=1e-03
  [  9/60] val_elbo=26.583  mse_ratio=6.047  z=3 h=32 β=0.1 anneal=150 ep=150 wd=0.0001 lr=1e-03
  [ 10/60] val_elbo=24.551  mse_ratio=1.349  z=1 h=32 β=1.0 anneal=100 ep=300 wd=0.001 lr=5e-04
 

## マルチシード実験（data_seed 0〜9 × model_seed 再挑戦）

採用設定（ハイパラ・z_dim）は固定し、data_seedごとに model_seed を独立に複数回振る（初期値運対策）。
学習は inner(80%)のみで行い、val(20%)は崩壊ガード信号（val_elbo, mse_ratio）専用の完全held-outとして残す。
真のτ・真のZは test set での採点（sqrt_pehe, z_recovery）にのみ使う。

In [ ]:
N_DATA_SEEDS = 10
N_RESTARTS = 5   # data_seedごとのmodel_seed再挑戦回数（v4の教訓：初期値運があるため複数回試す）

multi_records = []

for ds in range(N_DATA_SEEDS):
    print(f"\n=== data_seed={ds} ===", flush=True)
    Xs, Ts, Ys, _, _, taus, Zs = generate_toy_v5(seed=ds)

    df_s = pd.DataFrame(Xs, columns=proxy_cols)
    df_s["T"] = Ts; df_s["Y"] = Ys; df_s["tau"] = taus
    df_tr_s, df_te_s = train_test_split(df_s, test_size=0.3,
                                        random_state=ds, stratify=df_s["T"])
    tau_true_s = df_te_s["tau"].values          # ← 答え合わせ専用
    te_idx = df_te_s.index.tolist()
    Z_te_s = Zs[te_idx]                          # ← 答え合わせ専用

    # 学習用(inner)とガード信号計算用(val)を分離（valは学習に一切使わない、正真正銘のheld-out）
    df_inner_s, df_val_s = train_test_split(df_tr_s, test_size=0.2,
                                            random_state=ds, stratify=df_tr_s["T"])
    x_inner_s, t_inner_s, y_inner_s = df_to_tensors(df_inner_s)
    x_val_s,   t_val_s,   y_val_s   = df_to_tensors(df_val_s)
    y_val_s_np = df_val_s["Y"].values

    xt = torch.tensor(Xs, dtype=torch.float32)
    tt = torch.tensor(Ts, dtype=torch.float32)
    yt = torch.tensor(Ys, dtype=torch.float32)
    x_te_s, t_te_s, y_te_s = xt[te_idx], tt[te_idx], yt[te_idx]

    # ── X-learner（data_seed=0の固定パラメータで再学習）──
    m0_f, m1_f, x0_f, x1_f, _ = xl_models_fixed
    df0s = df_tr_s[df_tr_s["T"]==0]; df1s = df_tr_s[df_tr_s["T"]==1]
    m0s = m0_f.__class__(**m0_f.get_params()).fit(df0s[proxy_cols], df0s["Y"])
    m1s = m1_f.__class__(**m1_f.get_params()).fit(df1s[proxy_cols], df1s["Y"])
    D0s = m1s.predict(df0s[proxy_cols]) - df0s["Y"].values
    D1s = df1s["Y"].values - m0s.predict(df1s[proxy_cols])
    x0s = x0_f.__class__(**x0_f.get_params()).fit(df0s[proxy_cols], D0s)
    x1s = x1_f.__class__(**x1_f.get_params()).fit(df1s[proxy_cols], D1s)
    props = LogisticRegression(max_iter=1000).fit(df_tr_s[proxy_cols], df_tr_s["T"])
    g0s = props.predict_proba(df_te_s[proxy_cols])[:, 0]
    tau_xl = g0s * x0s.predict(df_te_s[proxy_cols]) + (1-g0s) * x1s.predict(df_te_s[proxy_cols])
    xl_pehe_s = float(np.sqrt(np.mean((tau_true_s - tau_xl)**2)))
    xl_eps_ate_s = float(abs(tau_xl.mean() - tau_true_s.mean()))

    # ── CEVAE：採用設定(BEST_CEVAE_CFG)を固定し、model_seedだけN_RESTARTS回振る ──
    for ms in range(N_RESTARTS):
        cfg_fields = {k: getattr(BEST_CEVAE_CFG, k) for k in BEST_CEVAE_CFG.__dataclass_fields__}
        cfg_fields["seed"] = ms
        cfg_ms = CEVAEConfig(**cfg_fields)
        nets_ms = build_cevae(cfg_ms, d)
        train_cevae(nets_ms, cfg_ms, x_inner_s, t_inner_s, y_inner_s)

        # ガード信号（真のτ・真のZを使わない。held-outのvalで計算）
        val_elbo_ms = compute_val_elbo(nets_ms, x_val_s, t_val_s, y_val_s)
        mse_q_ms, mse_p_ms, mse_ratio_ms = compute_y_mse(nets_ms, x_val_s, t_val_s, y_val_s_np)

        # 答え合わせ専用（真のτ・真のZを使う。testで計算）
        tau_pred_ms = estimate_ite(nets_ms, x_te_s)
        pehe_ms = float(np.sqrt(np.mean((tau_true_s - tau_pred_ms) ** 2)))
        eps_ate_ms = float(abs(tau_pred_ms.mean() - tau_true_s.mean()))
        zrec_ms = z_recovery_score(nets_ms, x_te_s, t_te_s, y_te_s, Z_te_s)

        multi_records.append({
            "data_seed": ds, "model_seed": ms, "z_dim": cfg_ms.z_dim,
            "val_elbo": val_elbo_ms, "mse_q": mse_q_ms, "mse_p": mse_p_ms, "mse_ratio": mse_ratio_ms,
            "sqrt_pehe": pehe_ms, "eps_ate": eps_ate_ms, "z_recovery": zrec_ms,
            "xl_pehe": xl_pehe_s, "xl_eps_ate": xl_eps_ate_s,
        })
        print(f"  ms={ms}  PEHE={pehe_ms:.3f}  z_recovery={zrec_ms:.3f}  "
              f"elbo={val_elbo_ms:.3f}  mse_ratio={mse_ratio_ms:.3f}")

df_multi = pd.DataFrame(multi_records)
display(df_multi.round(4))

## 崩壊ガードを適用してCEVAEの成績を集計

In [ ]:
GUARD_TH_MULTI = guard_threshold(df_multi["mse_ratio"])
df_multi["guard_pass"] = df_multi["mse_ratio"] <= GUARD_TH_MULTI
print(f"マルチシード実験の崩壊ガード閾値 = {GUARD_TH_MULTI:.3f}")
print(f"ガード通過: {int(df_multi['guard_pass'].sum())}/{len(df_multi)} run")
print(df_multi.groupby("data_seed")["guard_pass"].mean().rename("生存率"))

df_pass = df_multi[df_multi["guard_pass"]]

seeds_with_survivors = set(df_pass["data_seed"].unique())
seeds_all_collapsed = sorted(set(range(N_DATA_SEEDS)) - seeds_with_survivors)
if seeds_all_collapsed:
    print(f"\n警告: data_seed {seeds_all_collapsed} は全restartが崩壊ガードで除外されました（要確認）")

print("\n=== 崩壊ガード適用後のCEVAE成績（生存runのみ、mean±std） ===")
print(df_pass[["sqrt_pehe", "eps_ate", "z_recovery"]].agg(["mean", "std"]).round(3))

xl_summary_df = df_multi.drop_duplicates("data_seed")[["data_seed", "xl_pehe", "xl_eps_ate"]].set_index("data_seed")
print("\n=== X-learner成績（data_seedごとに1つ、mean±std） ===")
print(xl_summary_df.agg(["mean", "std"]).round(3))

# 真の崩壊ラベル（図・混同行列用。z_recoveryはR²なので0～1。下の散布図を見て必要なら調整）
Z_RECOVERY_SUCCESS_TH = 0.5
df_multi["collapsed_truth"] = df_multi["z_recovery"] < Z_RECOVERY_SUCCESS_TH
print(f"\n真の崩壊率（z_recovery<{Z_RECOVERY_SUCCESS_TH}）: "
      f"{int(df_multi['collapsed_truth'].sum())}/{len(df_multi)}")

## 散布図・混同行列：2次元でも mse_ratio で二山が分離するか

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 5))
for collapsed, color, label in [(False, "#2ecc71", "成功"), (True, "#e74c3c", "崩壊")]:
    sub = df_multi[df_multi["collapsed_truth"] == collapsed]
    ax.scatter(sub["mse_ratio"], sub["sqrt_pehe"], c=color, label=label,
               alpha=0.85, edgecolor="black", linewidth=0.4, s=60)
ax.axvline(GUARD_TH_MULTI, color="gray", linestyle="--", label=f"ガード閾値={GUARD_TH_MULTI:.2f}")
ax.set_xlabel("mse_ratio = mse_p / mse_q")
ax.set_ylabel("√PEHE")
ax.set_title("2次元Zでもmse_ratioで二山が分離するか")
ax.legend()
plt.tight_layout()
plt.show()


def confusion_at_threshold(df, signal_col, threshold, direction, true_col="collapsed_truth"):
    true_c = df[true_col]
    pred_c = df[signal_col] >= threshold if direction == "above" else df[signal_col] <= threshold
    tp = int((pred_c & true_c).sum()); fn = int((~pred_c & true_c).sum())
    fp = int((pred_c & ~true_c).sum()); tn = int((~pred_c & ~true_c).sum())
    return {"signal": signal_col, "direction": direction, "threshold": threshold,
            "崩壊を正しく検出(TP)": tp, "崩壊を見逃す(FN)": fn,
            "成功を誤って除外(FP)": fp, "成功を正しく残す(TN)": tn, "errors": fn + fp}


def best_threshold(df, signal_col, true_col="collapsed_truth"):
    values = np.sort(df[signal_col].unique())
    candidates = (values[:-1] + values[1:]) / 2 if len(values) > 1 else values
    best = None
    for direction in ["above", "below"]:
        for th in candidates:
            res = confusion_at_threshold(df, signal_col, th, direction, true_col)
            if best is None or res["errors"] < best["errors"]:
                best = res
    return best


print("各信号の「最良の閾値」での混同行列（val_elboがベースライン）\n")
comparison = pd.DataFrame([best_threshold(df_multi, c) for c in ["val_elbo", "mse_ratio"]])
display(comparison)

guard_actual = confusion_at_threshold(df_multi, "mse_ratio", GUARD_TH_MULTI, "above")
print("\n実際に採用したガード閾値（相対ルール由来）での混同行列:")
print(guard_actual)

## 本丸：崩壊ガード適用後 CEVAE vs X-learner、および1次元(v4)との対比

In [ ]:
print("=== 本丸：崩壊ガード適用後 CEVAE vs X-learner（√PEHE, ε_ATE） ===")
final_table = pd.DataFrame({
    "CEVAE(ガード後)": [df_pass["sqrt_pehe"].mean(), df_pass["sqrt_pehe"].std(),
                        df_pass["eps_ate"].mean(), df_pass["eps_ate"].std()],
    "X-learner":        [xl_summary_df["xl_pehe"].mean(), xl_summary_df["xl_pehe"].std(),
                        xl_summary_df["xl_eps_ate"].mean(), xl_summary_df["xl_eps_ate"].std()],
}, index=["sqrt_pehe_mean", "sqrt_pehe_std", "eps_ate_mean", "eps_ate_std"])
display(final_table.round(3))

cevae_mean = df_pass["sqrt_pehe"].mean()
xl_mean = xl_summary_df["xl_pehe"].mean()
print(f"\nCEVAE(ガード後、全体平均)がX-learnerを"
      f"{'下回った（勝ち）' if cevae_mean < xl_mean else '下回れなかった（負け）'}"
      f"（CEVAE={cevae_mean:.3f} vs XL={xl_mean:.3f}）")

# seedごとの勝敗（生存restartがあるseedのみ対象）
per_seed_cevae = df_pass.groupby("data_seed")["sqrt_pehe"].mean()
per_seed_xl = xl_summary_df["xl_pehe"]
common_seeds = per_seed_cevae.index.intersection(per_seed_xl.index)
n_win = int((per_seed_cevae.loc[common_seeds] < per_seed_xl.loc[common_seeds]).sum())
print(f"CEVAE勝率(seed別、ガード後平均で比較): {n_win}/{len(common_seeds)}")

# ── 1次元(v4)との対比表 ──
# v4(1次元Z, data_seed 0-9, 単一model_seed=data_seed, ガード無し)の既知の結果を参考値として併記。
ref_1d = pd.DataFrame({
    "指標": ["崩壊率(/全run)", "CEVAE √PEHE (mean±std)", "X-learner √PEHE (mean±std)", "CEVAE勝率"],
    "1次元(v4, 参考値、ガード無し・単一restart)": ["3/10", "3.271±2.818", "2.315±0.228", "7/10"],
    "2次元(v5, 本ノート、ガード有り・複数restart)": [
        f"{int((~df_multi['guard_pass']).sum())}/{len(df_multi)}",
        f"{df_pass['sqrt_pehe'].mean():.3f}±{df_pass['sqrt_pehe'].std():.3f}",
        f"{xl_summary_df['xl_pehe'].mean():.3f}±{xl_summary_df['xl_pehe'].std():.3f}",
        f"{n_win}/{len(common_seeds)}",
    ],
})
display(ref_1d)